<a href="https://colab.research.google.com/github/Brant-c/chess-opening-ml/blob/main/LichessEDA%5BG66%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Milestone 2: Dataset and Problem Approval
**Project Title:** Personalized Chess Opening Recommendations Using Machine Learning  
**Team Name:** Machine Learners (Team 66)
---

## 1. Data Source & License

* **Dataset Source:** [Lichess Open Database](https://database.lichess.org/)
* **Dataset Download Link:** [August 2016 Standard Rated Games](https://database.lichess.org/standard/lichess_db_standard_rated_2016-08.pgn.zst)
* **License:** Released into the public domain under the [Creative Commons CC0 1.0 Universal (CC0 1.0) Public Domain Dedication](https://creativecommons.org/publicdomain/zero/1.0/).
* **Usage & Rights:** The CC0 dedication waives all copyright rights worldwide. We have full permission to parse, clean, engineer features, store extracted tabular subsets, and distribute model artifacts for academic and non-commercial research purposes.

---

## 2. Data Card

* **Dataset Overview:** Contains chess game records from 6,483,257 standard rated games played on Lichess.org during August 2016.
* **Key Attributes Extracted:** Player Elo ratings (`WhiteElo`, `BlackElo`), match time control (`TimeControl`), opening identifiers (`ECO`, `Opening`), and game outcome (`Result`).
* **Target Audience & Purpose:** Used to train predictive models that estimate game outcome probabilities, which are then used by our recommendation system to give personalized opening advice to players based on their skill bracket.
* **Data Cleaning & Filtering:** High-cardinality metadata (`White`, `Black`, `Site`, `Event`, `Round`, `UTCTime`) and post-game leakage features (`WhiteRatingDiff`, `BlackRatingDiff`, `WhiteTitle`, `BlackTitle`) are removed prior to training.

---

## 3. Inputs, Outputs, and Task Definition

### Target & Task Definition
* **Machine Learning Task:** Supervised Multi-Class Classification.
* **Target Variable ($Y$):** `result` (Categorical outcome of the game).
  * `1` : White Win
  * `0` : Draw
  * `-1` : Black Win
* **Model Objective:** Estimate the conditional probability distribution $P(Y \mid X)$ across all three potential outcomes given player ratings, match conditions, and opening choices.
* **Downstream Application (Recommendation Engine):** The model takes user query inputs, evaluates candidate openings (`ECO` codes) through the trained classifier, and outputs a ranked list of openings that maximize the user's expected win/draw likelihood for their Elo tier.

### Sample Feature Vector ($X$) & Model Outputs ($Y$)
* **Model Input Feature Vector ($X$):**
  `[avg_elo=1450, elo_diff=35, time_control_class="Rapid", eco_code="C50", opening_draw_rate=0.22]`
* **Model Probabilistic Output ($Y$):**
  `[P(White Win)=0.54, P(Draw)=0.26, P(Black Win)=0.20]`

### User Interface Query Inputs
When a chess player uses the recommendation interface, they do not enter internal model features like `eco_code` or `elo_diff`. Instead, they enter:
1. `user_elo` (e.g., `1400`)
2. `time_control_class` (e.g., `"Rapid"`)
3. `playstyle_preference` (Optional: e.g., `"Tactical"`, `"Aggressive"`, or `"Solid"`)

---

In [1]:
# Chess Reading Tools
!pip install python-chess zstandard -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 37.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### **2.1 Import Dataset + Tools**


In [2]:
import pandas as pd
import chess
import chess.pgn
import zstandard as zstd
import io



# file_path = "lichess_db_standard_rated_2014-08.pgn.zst"
file_path = "lichess_db_standard_rated_2016-08.pgn.zst"

rows = []

with open(file_path, "rb") as f:
    reader = zstd.ZstdDecompressor().stream_reader(f)
    text = io.TextIOWrapper(reader)

    for i in range(10000):
        game = chess.pgn.read_game(text)

        if game is None:
            break

        rows.append(dict(game.headers))

df = pd.DataFrame(rows)

### **2.2 Data Description**


In [3]:
df.shape

(10000, 19)

In [4]:
df.describe()

,Event,Site,Date,Round,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination,WhiteTitle,BlackTitle
count,10000,10000,10000,10000,10000,10000,10000,10000,10000,10000,10000,9997,9997,10000,10000,10000,10000,23,26
unique,17,10000,1,1,4428,4459,3,1,3476,1329,1357,244,257,339,1204,201,4,4,4
top,Rated Blitz game,https://lichess.org/lgCj0qfj,????.??.??,?,Solarspion,Solarspion,1-0,2016.07.31,23:00:01,1500,1500,+10,-10,A00,Van't Kruijs Opening,300+0,Normal,FM,FM
freq,3859,1,10000,10000,41,41,4928,10000,38,81,53,433,441,654,204,1806,6843,11,12


In [5]:
df.isnull().sum()

,0
Event,0
Site,0
Date,0
Round,0
White,0
Black,0
Result,0
UTCDate,0
UTCTime,0
WhiteElo,0


In [6]:
df[["ECO", "Opening"]].head(20)

,ECO,Opening
0,?,?
1,B00,King's Pawn
2,B00,"St. George Defense: New St. George, Three Pawn..."
3,C53,"Italian Game: Classical Variation, Albin Gambit"
4,E90,King's Indian Defense: Larsen Variation
5,A46,Torre Attack #2
6,D15,Slav Defense: Three Knights Variation
7,B06,Modern Defense: Two Knights Variation
8,B30,Sicilian Defense: Old Sicilian
9,A40,Englund Gambit Complex: Hartlaub-Charlick Gambit


In [7]:
df["ECO"].value_counts()

,count
ECO,
A00,654
A40,491
B01,490
C00,473
D00,368
...,...
C95,1
A27,1
E80,1


In [8]:
df["Event"].value_counts().head(20)

,count
Event,
Rated Blitz game,3859
Rated Classical game,2189
Rated Bullet game,1908
Rated Classical tournament https://lichess.org/tournament/LGcu43Jf,372
Rated Bullet tournament https://lichess.org/tournament/RRKqM84J,368
Rated Blitz tournament https://lichess.org/tournament/aoErwkcR,368
Rated Bullet tournament https://lichess.org/tournament/noX0h0VB,338
Rated Blitz tournament https://lichess.org/tournament/CDDuycev,290
Rated Blitz tournament https://lichess.org/tournament/encooMGt,69


### **2.3 Data Cleaning**

#### **Dropped Fields:**
- Site
- Date
- Round
- White
- Black
- UTCTime
- WhiteRatingDiff
- BlackRatingDiff
- WhiteTitle
- BlackTitle

#### **Persistent Fields:**
1.   Event
2.   Result - W/L for who
3.   BlackElo
4.   ECO
5.   *Opening
6.   TimeControl (420 + 8 = 420s per player + 8s per turn)
7.   Termination
8.   Moves

***Note:** Opening and ECO represent the same thing, opening can be more accurate in some instances but is less consistent than ECO



In [9]:
columns_to_drop = [
    "Site",
    "Date",
    "Round",
    "White",
    "Black",
    "UTCTime",
    "UTCDate",
    #"WhiteRatingDiff",
    "BlackRatingDiff", # White always goes first and every row has a rating for W/B (Not sure why you would need both?)
    "WhiteTitle",
    "BlackTitle"
]

dfclean = df.drop(columns=columns_to_drop)

dfclean.head(50)

,Event,Result,WhiteElo,BlackElo,WhiteRatingDiff,ECO,Opening,TimeControl,Termination
0,Rated Bullet tournament https://lichess.org/to...,0-1,1801,1813,-12,?,?,60+0,Abandoned
1,Rated Blitz tournament https://lichess.org/tou...,1-0,1500,1476,+163,B00,King's Pawn,300+0,Abandoned
2,Rated Blitz tournament https://lichess.org/tou...,0-1,1261,1345,-11,B00,"St. George Defense: New St. George, Three Pawn...",300+0,Normal
3,Rated Bullet game,0-1,1744,1848,-7,C53,"Italian Game: Classical Variation, Albin Gambit",120+1,Time forfeit
4,Rated Bullet tournament https://lichess.org/to...,1-0,1485,1471,+15,E90,King's Indian Defense: Larsen Variation,60+0,Time forfeit
5,Rated Bullet tournament https://lichess.org/to...,0-1,2107,2132,-11,A46,Torre Attack #2,60+0,Normal
6,Rated Blitz tournament https://lichess.org/tou...,1-0,1598,1525,+9,D15,Slav Defense: Three Knights Variation,300+0,Normal
7,Rated Bullet tournament https://lichess.org/to...,0-1,1869,1903,-11,B06,Modern Defense: Two Knights Variation,60+0,Normal
8,Rated Blitz tournament https://lichess.org/tou...,0-1,1500,1506,-172,B30,Sicilian Defense: Old Sicilian,300+0,Normal
9,Rated Bullet game,0-1,2171,2254,-9,A40,Englund Gambit Complex: Hartlaub-Charlick Gambit,30+0,Time forfeit
